In [71]:
import getpass
import os

import psycopg2
from langchain.agents import create_agent
from langchain.messages import AIMessageChunk
from langchain_google_genai import ChatGoogleGenerativeAI

In [57]:
DB_NAME: str = os.getenv("DB_NAME", "postgres")
DB_USER: str = os.getenv("DB_USER", "postgres")
DB_PASSWORD: str = os.getenv("DB_PASSWORD", "postgres")
DB_HOST: str = os.getenv("DB_HOST", "localhost")
DB_PORT: str = os.getenv("DB_PORT", "5432")

conn = psycopg2.connect(
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
)

In [58]:
def list_tables():
    """
    List all available tables in the public schema.

    This tool is useful to explore the database structure before querying data.

    Returns:
        list[str]: Names of all tables in the public schema.
    """
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
        """)
        return [r[0] for r in cur.fetchall()]


def describe_table(table: str):
    """
    Describe the structure of a given table.

    Use this tool to understand column names and types before writing queries.

    Args:
        table (str): Name of the table to inspect.

    Returns:
        list[tuple]: Each tuple contains (column_name, data_type).
    """
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_name = %s
        """,
            (table,),
        )
        return cur.fetchall()


def run_query(sql: str):
    """
    Execute a read-only SQL query on the database.

    Only SELECT queries are allowed. Any query containing write operations
    (INSERT, UPDATE, DELETE, DROP, TRUNCATE) will be blocked.

    Use this tool to retrieve data after exploring tables and columns.

    Args:
        sql (str): A SQL SELECT query.

    Returns:
        list[tuple] | str:
            - Query results if rows are returned
            - "Query executed" if no rows are returned
            - Error message if the query is blocked
    """
    # ⚠️ sécurité minimale
    forbidden = ["drop", "delete", "truncate", "update", "insert"]
    if any(word in sql.lower() for word in forbidden):
        return "Write queries are blocked"

    with conn.cursor() as cur:
        cur.execute(sql)

        if cur.description:
            return cur.fetchall()
        return "Query executed"

In [59]:
list_tables()

['staging_raw',
 'mean_bicycle_available_by_station',
 'consolidate_station_statement',
 'consolidate_city',
 'map_station',
 'dim_city',
 'consolidate_station',
 'available_emplacement_by_city',
 'dim_station',
 'fact_station_statement',
 'total_capacity_by_city']

In [60]:
describe_table("total_capacity_by_city")

[('total_capacity', 'bigint'), ('city_name', 'text')]

In [61]:
api_key = getpass.getpass("Enter your Google AI API key: ")

In [65]:
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", api_key=api_key)

In [66]:
agent = create_agent(
    model=model,
    tools=[list_tables, describe_table, run_query],
    system_prompt="You are a helpful assistant",
)

In [67]:
res = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Quels est  le nombre total de places libres par ville ?",
            }
        ]
    }
)

In [68]:
for i in res["messages"]:
    print(i, end="\n")

content='Quels est  le nombre total de places libres par ville ?' additional_kwargs={} response_metadata={} id='33662e88-12f8-4323-ac37-a7b6004331d5'
content=[] additional_kwargs={'function_call': {'name': 'list_tables', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'621efe20-e580-4ae6-8f33-058ae51ed6b5': 'EuYBCuMBAb4+9vvd0gQilO3P30CAq4BbRk7ZGKeTCAOGTDGcCj5fQI/TJ6GPEatogZ2NCCEx57mlyEsyyhijdAzJpiUgGn3wGrQ7GNWlAr7WOWgVOHSgCRKic/jK8UNE+PFA2heTRKQTzoDEuADu0J0redyq2ADvBLopiMIZpPKgy+bYyu1QxIolKdTZjioim/maNhSJo96i0Z/qiLAl61s/l0hUJ44DY4ZYVoK79n9zg8D717UIKi7xeoNo/AxYqnV+bhdsER11jbyVvRgbwBu+zfSeBH9c2IYzciEQrOUqEEMVqYs9BTo='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019d1c55-b565-78d0-a3fc-08853b1c9dfd-0' tool_calls=[{'name': 'list_tables', 'args': {}, 'id': '621efe20-e580-4ae6-8f33-058ae51ed6b5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_t

In [69]:
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Quels est  le nombre total de places libres par ville ?",
            }
        ]
    },
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'id': '1203f465-b726-4769-832c-84dea18789ff', 'name': 'list_tables', 'args': {}}]
step: tools
content: [{'type': 'text', 'text': '["staging_raw", "mean_bicycle_available_by_station", "consolidate_station_statement", "consolidate_city", "map_station", "dim_city", "consolidate_station", "available_emplacement_by_city", "dim_station", "fact_station_statement", "total_capacity_by_city"]'}]
step: model
content: [{'type': 'tool_call', 'id': 'e8f397ec-6e68-4aa3-aa18-515560eb0090', 'name': 'describe_table', 'args': {'table': 'available_emplacement_by_city'}}]
step: tools
content: [{'type': 'text', 'text': '[["sum_bicycle_docks_available", "bigint"], ["name", "text"]]'}]
step: model
content: [{'type': 'tool_call', 'id': '0096f271-48b1-41fc-ae02-7937cb298e23', 'name': 'run_query', 'args': {'sql': 'SELECT name, sum_bicycle_docks_available FROM available_emplacement_by_city;'}}]
step: tools
content: [{'type': 'text', 'text': '[["strasbourg", 219], ["nant

In [73]:
for token, metadata in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Quels est  le nombre total de places libres par ville ?",
            }
        ]
    },
    stream_mode="messages",
):
    if not isinstance(token, AIMessageChunk):
        continue
    reasoning = [b for b in token.content_blocks if b["type"] == "reasoning"]
    text = [b for b in token.content_blocks if b["type"] == "text"]
    if reasoning:
        print(f"[thinking] {reasoning[0]['reasoning']}", end="")
    if text:
        print(text[0]["text"], end="")

Voici le nombre total de places libres (bornes disponibles) par ville :

*   **Paris** : 20 100 places
*   **Nantes** : 1 429 places
*   **Strasbourg** : 219 places